In [1]:
# Imports
import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

from pathlib import Path




In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

In [3]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


In [4]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [5]:

X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 


Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


In [6]:
from tensorflow.keras import regularizers

num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomTranslation(0.03, 0.03),

    tf.keras.layers.Conv2D(
        32, (3,3), activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(0.001)
    ),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(
        64, (3,3),
        activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(0.001)
    ),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(
        128, (3,3),
        activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(0.001)
    ),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        128, activation='relu',
        kernel_regularizer=regularizers.l2(0.001)
    ),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 224, 224, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model_baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

baseline_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODELS_DIR / "baseline_best.keras"),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

In [9]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=baseline_callbacks
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3016 - loss: 1.6263
Epoch 1: val_accuracy improved from None to 0.23858, saving model to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_best.keras

Epoch 1: finished saving model to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_best.keras
90/90 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.2948 - loss: 1.6150 - val_accuracy: 0.2386 - val_loss: 1.6159 - learning_rate: 1.0000e-04
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2934 - loss: 1.5781
Epoch 2: val_accuracy did not improve from 0.23858
90/90 ━━━━━━━━━━━━━━━━━━━━ 173s 2s/step - accuracy: 0.2962 - loss: 1.5690 - val_accuracy: 0.1878 - val_loss: 1.6059 - learning_rate: 1.0000e-04
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.3311 - loss: 1.5365 
Epoch 3: val_accuracy did not improve from 0.23858
90/90 ━━━━━━━━━━━━━━━━━━━━ 1163s 13s/step - accuracy: 0.3404 - loss: 1.5278 - val_accuracy: 0.2259 - val_loss: 1.58

In [10]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 378ms/step - accuracy: 0.2259 - loss: 1.5844
Baseline Loss: 1.5844318866729736
Baseline Accuracy: 0.22588832676410675


In [11]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 371ms/step
Confusion Matrix:
 [[19 23  0 58]
 [ 4 17  0 94]
 [23 36  0 46]
 [ 6 15  0 53]]

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.37      0.19      0.25       100
meningioma_tumor       0.19      0.15      0.17       115
        no_tumor       0.00      0.00      0.00       105
 pituitary_tumor       0.21      0.72      0.33        74

        accuracy                           0.23       394
       macro avg       0.19      0.26      0.19       394
    weighted avg       0.19      0.23      0.17       394



d:\SE\SP\SeniorProject\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\SE\SP\SeniorProject\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\SE\SP\SeniorProject\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
os.makedirs(MODELS_DIR, exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
model_baseline.save(str(baseline_model_path))
print(f"Saved to {baseline_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_cnn.keras


# Transfer Learning Generators

In [13]:
transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [14]:
from tensorflow.keras import regularizers

base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(
    128,
    activation='relu',
    kernel_regularizer=regularizers.l2(0.0005)
    )(X)
X=Dropout(0.30)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)

model_transfer=Model(inputs=base_model.input, outputs=outputs)
model_transfer.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# Compile

In [15]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
transfer_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODELS_DIR / "mobilenetv2_best.keras"),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

In [17]:
EPOCHS_TL = 15

history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL,
    callbacks=transfer_callbacks
)

Epoch 1/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.4273 - loss: 1.4330
Epoch 1: val_accuracy improved from None to 0.47462, saving model to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_best.keras

Epoch 1: finished saving model to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_best.keras
90/90 ━━━━━━━━━━━━━━━━━━━━ 88s 944ms/step - accuracy: 0.5369 - loss: 1.2021 - val_accuracy: 0.4746 - val_loss: 1.3084 - learning_rate: 1.0000e-04
Epoch 2/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 815ms/step - accuracy: 0.7184 - loss: 0.8193
Epoch 2: val_accuracy improved from 0.47462 to 0.54061, saving model to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_best.keras

Epoch 2: finished saving model to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_best.keras
90/90 ━━━━━━━━━━━━━━━━━━━━ 82s 906ms/step - accuracy: 0.7268 - loss: 0.7947 - val_accuracy: 0.5406 - val_loss: 1.2698 - learning_rate: 1.0000e-04
Epoch 3/15
90/90 ━━━━━━━━━━━━━━━━

In [18]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 605ms/step - accuracy: 0.6904 - loss: 1.1361
Transfer Learning Loss: 1.1360950469970703


In [19]:
print('Comparision:')
print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

Comparision:
Baseline Accuracy: 0.22588832676410675
Transfer Learning Accuracy: 0.6903553009033203


In [20]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10,
    callbacks=transfer_callbacks
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 936ms/step - accuracy: 0.6348 - loss: 1.0320
Epoch 1: val_accuracy did not improve from 0.70051
90/90 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.7192 - loss: 0.8327 - val_accuracy: 0.6827 - val_loss: 1.1559 - learning_rate: 1.0000e-05
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 944ms/step - accuracy: 0.8320 - loss: 0.5722
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.

Epoch 2: val_accuracy did not improve from 0.70051
90/90 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.8369 - loss: 0.5466 - val_accuracy: 0.6726 - val_loss: 1.1572 - learning_rate: 1.0000e-05
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 935ms/step - accuracy: 0.8633 - loss: 0.4776
Epoch 3: val_accuracy did not improve from 0.70051
90/90 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.8551 - loss: 0.4946 - val_accuracy: 0.6777 - val_loss: 1.1443 - learning_rate: 5.0000e-06
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8626 - loss: 0.4709


In [21]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_transfer.keras
